In [1]:
import pandas as pd

train_df = pd.read_csv(
    "../data/processed/train_df.csv"
)

test_df = pd.read_csv(
    "../data/processed/test_df.csv"
)

print(train_df.shape)
print(test_df.shape)

(101908, 5)
(6197, 5)


In [2]:
user_history = (
    train_df
    .groupby("user_idx")["book_idx"]
    .apply(set)
    .to_dict()
)

In [3]:
print(len(user_history))

6197


In [5]:
n_users = max(
    train_df["user_idx"].max(),
    test_df["user_idx"].max()
) + 1

n_books = max(
    train_df["book_idx"].max(),
    test_df["book_idx"].max()
) + 1

print("Users:", n_users)
print("Books:", n_books)

Users: 6436
Books: 9009


In [9]:
print(train_df["user_idx"].nunique())
print(test_df["user_idx"].nunique())

6197
6197


In [10]:
user_history = (
    train_df
    .groupby("user_idx")["book_idx"]
    .apply(set)
    .to_dict()
)

print(len(user_history))

6197


In [11]:
import torch
import random

from torch.utils.data import Dataset
class BPRDataset(Dataset):

    def __init__(self, interactions, user_history, n_books):

        self.users = interactions["user_idx"].values
        self.items = interactions["book_idx"].values

        self.user_history = user_history
        self.n_books = n_books

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):

        user = self.users[idx]
        pos = self.items[idx]

        while True:

            neg = random.randint(0, self.n_books - 1)

            if neg not in self.user_history[user]:
                break

        return (
            torch.tensor(user),
            torch.tensor(pos),
            torch.tensor(neg)
        )
          

In [12]:
bpr_dataset = BPRDataset(
    train_df,
    user_history,
    n_books
)

In [13]:
u, p, n = bpr_dataset[0]

print("User:", u.item())
print("Positive:", p.item())
print("Negative:", n.item())

User: 0
Positive: 1
Negative: 632


In [14]:
u, p, n = bpr_dataset[0]

print(p.item() in user_history[u.item()])
print(n.item() in user_history[u.item()])

True
False


In [15]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    bpr_dataset,
    batch_size=1024,
    shuffle=True,
    pin_memory=True,
    num_workers=0,
)

In [16]:
print(len(train_loader))

100


In [17]:
print(len(bpr_dataset))

101908


In [18]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("Device Count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Torch Version:", torch.__version__)
print("CUDA Version:", torch.version.cuda)

CUDA Available: True
Device Count: 1
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Torch Version: 2.11.0+cu128
CUDA Version: 12.8


In [19]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [20]:
users, pos_items, neg_items = next(iter(train_loader))

In [21]:
print(users.shape)
print(pos_items.shape)
print(neg_items.shape)

torch.Size([1024])
torch.Size([1024])
torch.Size([1024])


In [22]:
import torch
import torch.nn as nn

class BPRModel(nn.Module):

    def __init__(self, n_users, n_books, embedding_dim=50):

        super().__init__()

        self.user_embedding = nn.Embedding(
            n_users,
            embedding_dim
        )

        self.book_embedding = nn.Embedding(
            n_books,
            embedding_dim
        )

        nn.init.normal_(
            self.user_embedding.weight,
            std=0.01
        )

        nn.init.normal_(
            self.book_embedding.weight,
            std=0.01
        )

    def forward(self, users, items):

        user_vecs = self.user_embedding(users)

        item_vecs = self.book_embedding(items)

        scores = (user_vecs * item_vecs).sum(dim=1)

        return scores

In [23]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for BPR training in this notebook.")

device = torch.device("cuda")

model = BPRModel(
    n_users=n_users,
    n_books=n_books,
    embedding_dim=50
).to(device)

print(model)
print(torch.cuda.get_device_name(0))

BPRModel(
  (user_embedding): Embedding(6436, 50)
  (book_embedding): Embedding(9009, 50)
)
NVIDIA GeForce RTX 4060 Laptop GPU


In [24]:
users, pos_items, neg_items = next(iter(train_loader))

users = users.to(device)
pos_items = pos_items.to(device)

scores = model(users, pos_items)

print(scores.shape)

torch.Size([1024])


In [25]:
import torch.nn.functional as F

def bpr_loss(pos_scores, neg_scores):

    loss = -torch.log(
        torch.sigmoid(pos_scores - neg_scores)
        + 1e-8
    ).mean()

    return loss

In [26]:
users, pos_items, neg_items = next(iter(train_loader))

users = users.to(device)
pos_items = pos_items.to(device)
neg_items = neg_items.to(device)

pos_scores = model(users, pos_items)
neg_scores = model(users, neg_items)

loss = bpr_loss(pos_scores, neg_scores)

print(loss.item())

0.6931337118148804


In [27]:
import torch.optim as optim

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

In [28]:
epochs = 100

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for users, pos_items, neg_items in train_loader:

        users = users.to(device, non_blocking=True)
        pos_items = pos_items.to(device, non_blocking=True)
        neg_items = neg_items.to(device, non_blocking=True)

        pos_scores = model(users, pos_items)

        neg_scores = model(users, neg_items)

        loss = bpr_loss(
            pos_scores,
            neg_scores
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1:02d} | Loss = {avg_loss:.4f}"
    )

Epoch 01 | Loss = 0.6931
Epoch 02 | Loss = 0.6908
Epoch 03 | Loss = 0.6723
Epoch 04 | Loss = 0.6262
Epoch 05 | Loss = 0.5745
Epoch 06 | Loss = 0.5337
Epoch 07 | Loss = 0.5011
Epoch 08 | Loss = 0.4740
Epoch 09 | Loss = 0.4523
Epoch 10 | Loss = 0.4334
Epoch 11 | Loss = 0.4164
Epoch 12 | Loss = 0.4027
Epoch 13 | Loss = 0.3892
Epoch 14 | Loss = 0.3780
Epoch 15 | Loss = 0.3656
Epoch 16 | Loss = 0.3566
Epoch 17 | Loss = 0.3471
Epoch 18 | Loss = 0.3368
Epoch 19 | Loss = 0.3291
Epoch 20 | Loss = 0.3231
Epoch 21 | Loss = 0.3171
Epoch 22 | Loss = 0.3079
Epoch 23 | Loss = 0.3031
Epoch 24 | Loss = 0.2963
Epoch 25 | Loss = 0.2912
Epoch 26 | Loss = 0.2860
Epoch 27 | Loss = 0.2799
Epoch 28 | Loss = 0.2763
Epoch 29 | Loss = 0.2704
Epoch 30 | Loss = 0.2670
Epoch 31 | Loss = 0.2630
Epoch 32 | Loss = 0.2589
Epoch 33 | Loss = 0.2560
Epoch 34 | Loss = 0.2518
Epoch 35 | Loss = 0.2483
Epoch 36 | Loss = 0.2456
Epoch 37 | Loss = 0.2415
Epoch 38 | Loss = 0.2390
Epoch 39 | Loss = 0.2356
Epoch 40 | Loss = 0.2327


In [29]:
with torch.no_grad():

    user_norms = (
        model.user_embedding.weight.norm(dim=1)
        .cpu()
        .numpy()
    )

    book_norms = (
        model.book_embedding.weight.norm(dim=1)
        .cpu()
        .numpy()
    )

print("User Norm Mean:", user_norms.mean())
print("Book Norm Mean:", book_norms.mean())

User Norm Mean: 1.4809883
Book Norm Mean: 1.370474


In [30]:
import numpy as np
import torch

def recommend_bpr(user_idx, model, user_history,
                  n_books, top_k=10):

    model.eval()

    with torch.no_grad():

        user_tensor = torch.tensor(
            [user_idx],
            device=device
        )

        user_emb = model.user_embedding(
            user_tensor
        )

        all_books = model.book_embedding.weight

        scores = torch.matmul(
            user_emb,
            all_books.T
        ).squeeze()

        scores = scores.cpu().numpy()

    seen_books = user_history[user_idx]

    scores[list(seen_books)] = -np.inf

    top_items = np.argsort(scores)[-top_k:][::-1]

    return top_items

In [31]:
recommend_bpr(
    user_idx=0,
    model=model,
    user_history=user_history,
    n_books=n_books,
    top_k=10
)

array([ 366,  431,  496,  683, 1833,  502, 1619, 1297,  626,  981])

In [32]:
recs = recommend_bpr(
    0,
    model,
    user_history,
    n_books,
    top_k=10
)

print(any(book in user_history[0] for book in recs))

False


In [33]:
def evaluate_hit10_bpr(
    model,
    test_df,
    user_history,
    n_books
):

    hits = 0
    total = len(test_df)

    for _, row in test_df.iterrows():

        user = row["user_idx"]
        true_book = row["book_idx"]

        recs = recommend_bpr(
            user,
            model,
            user_history,
            n_books,
            top_k=10
        )

        if true_book in recs:
            hits += 1

    hit10 = hits / total

    return hits, hit10

In [34]:
hits, hit10 = evaluate_hit10_bpr(
    model,
    test_df,
    user_history,
    n_books
)

print("Hits:", hits)
print("Hit@10:", hit10)

Hits: 320
Hit@10: 0.05163788930127481


In [35]:
torch.save(
    model.state_dict(),
    "../models/bpr_model.pth"
)

print("Saved!")

Saved!


In [36]:
torch.save(
    {
        "n_users": n_users,
        "n_books": n_books,
        "embedding_dim": 50
    },
    "../models/bpr_config.pth"
)

In [37]:
train_df.to_csv(
    "../data/processed/train_df.csv",
    index=False
)

test_df.to_csv(
    "../data/processed/test_df.csv",
    index=False
)

In [39]:
torch.save(
    model.state_dict(),
    "../models/bpr_model_cfsplit.pth"
)

torch.save(
    {
        "n_users": n_users,
        "n_books": n_books,
        "embedding_dim": 50
    },
    "../models/bpr_config_cfsplit.pth"
)